# Loading and Preprocessing

In [96]:
from data_loader import load_data
from decorrelation import keep_uncorrelated_features
from model_eval import result_viewer

In [97]:
patients,\
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [98]:
datasets = {
    "Patients": patients,
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

# Feature Selection
Dropping highly correlated clinical and radiomics features using the EDA analysis, as they add little to no new information.

## Highly Correlated Clinical Features  
We're using the EDA because it took the mixed feature types into account when calculating the correlations, and there are only a few features, so we could manually write them. So, it was more straight-forward than redoing it ourselves.

In [99]:
clinical_ftrs_to_drop = [
                'cli_st_neutrophils',
                'indication_dis_diagnosis.factor_DLBCL',
                'indication_dis_diagnosis.factor_tFL',
                'indication_extran_invol',
                'indication_extranodal_nr',
                'indication_pri_refr',
                'indication_sec_refr',
                'indication_age_60',
                'scr_weight',
                'scr_height',
                'tr_car_bridg_type.factor_None'
                ]

In [100]:
for name, df in datasets.items():
    print(f"Before dropping features in {name}: {df.shape}")
    df.drop(columns=clinical_ftrs_to_drop, inplace=True)
    print(f"After dropping features in {name}: {df.shape}")

Before dropping features in Patients: (86, 332)
After dropping features in Patients: (86, 321)
Before dropping features in Clinical: (86, 35)
After dropping features in Clinical: (86, 24)
Before dropping features in Clinical_A: (86, 134)
After dropping features in Clinical_A: (86, 123)
Before dropping features in Clinical_B: (86, 134)
After dropping features in Clinical_B: (86, 123)
Before dropping features in Clinical_Delta: (86, 134)
After dropping features in Clinical_Delta: (86, 123)


## Highly Correlated Radiomic Features
These are all numerical values, with high dimensionality. The EDA already showed a large number of highly correlated features, so we will write code to automatically keep only one of possibly multiple highly correlated features.

In [101]:
protected_features = {'SUV_Maximum_A', 'SUV_Mean_A', 'TLG_A', 'SUV_Maximum_B', 'SUV_Mean_B', 'TLG_B', 'SUV_Maximum_Delta', 'SUV_Mean_Delta', 'TLG_Delta'}
for name, df in datasets.items():
    print(f"Before dropping correlated features in {name}: {df.shape}")

    datasets[name] = keep_uncorrelated_features(df, threshold=0.8, protected_features=protected_features)
    print(f"After dropping correlated features in {name}: {datasets[name].shape}")

Before dropping correlated features in Patients: (86, 321)
After dropping correlated features in Patients: (86, 116)
Before dropping correlated features in Clinical: (86, 24)
After dropping correlated features in Clinical: (86, 24)
Before dropping correlated features in Clinical_A: (86, 123)
After dropping correlated features in Clinical_A: (86, 51)
Before dropping correlated features in Clinical_B: (86, 123)
After dropping correlated features in Clinical_B: (86, 51)
Before dropping correlated features in Clinical_Delta: (86, 123)
After dropping correlated features in Clinical_Delta: (86, 62)


In [102]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [103]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Creating a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)


We will try a suit of feature selection methods next.

1) Univariate feature selection: SelectKBest
2) Recursive feature elimination: RFECV
3) Select From Model: L1-based feature selection
4) Sequential Feature Selection

Uncomment to test each method

In [104]:
# SELECTKBEST WITH MUTUAL INFORMATION
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

fs = SelectKBest(mutual_info_classif, k=15)

# RFECV WITH LOGISTIC REGRESSION
# estimator = LogisticRegression(random_state=42)
# fs = RFECV(estimator, step=1, cv=5, scoring='roc_auc')

# LASSO WITH LOGISTIC REGRESSION
# estimator = LogisticRegression(penalty='l1', solver='liblinear', random_state=42)
# fs = SelectFromModel(estimator, max_features=10, threshold='median')

# SEQUENTIAL FEATURE SELECTION WITH LOGISTIC REGRESSION
# estimator = LogisticRegression(random_state=42)
# fs = SequentialFeatureSelector(estimator, n_features_to_select='auto', tol=0.01, direction='forward', cv=5, scoring='roc_auc')

In [105]:
models = {
    "Logistic Regression": LogisticRegression(C=0.8, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestClassifier(max_depth=5, min_samples_leaf=5, random_state=42)}

# ICANS 0,1

In [106]:
y = targets['ae_summ_icans_v2']

In [107]:
outcome = result_viewer(datasets, y, preprocessor, fs,models=models)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4204 ± 0.1381,0.5727 ± 0.1544,0.5376 ± 0.1642,0.5673 ± 0.1439,0.5228 ± 0.1249
train_roc_auc,0.7995 ± 0.0568,0.8115 ± 0.0398,0.8137 ± 0.0367,0.8185 ± 0.0281,0.8025 ± 0.0374
test_precision,0.5040 ± 0.1015,0.6213 ± 0.1347,0.5822 ± 0.1494,0.6114 ± 0.1141,0.5718 ± 0.1083
train_precision,0.7339 ± 0.0598,0.7411 ± 0.0445,0.7358 ± 0.0373,0.7609 ± 0.0381,0.7414 ± 0.0510
test_recall,0.5772 ± 0.1677,0.6756 ± 0.1526,0.6400 ± 0.1646,0.6283 ± 0.1617,0.6567 ± 0.1554
train_recall,0.8293 ± 0.0421,0.8385 ± 0.0401,0.8215 ± 0.0532,0.8425 ± 0.0344,0.8240 ± 0.0567
test_f1,0.5336 ± 0.1256,0.6414 ± 0.1268,0.6013 ± 0.1320,0.6121 ± 0.1212,0.6045 ± 0.1107
train_f1,0.7776 ± 0.0456,0.7862 ± 0.0367,0.7757 ± 0.0391,0.7985 ± 0.0221,0.7795 ± 0.0450


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4940 ± 0.1205,0.5460 ± 0.1506,0.5616 ± 0.1191,0.5596 ± 0.1587,0.4966 ± 0.1218
train_roc_auc,0.9359 ± 0.0212,0.9153 ± 0.0251,0.9233 ± 0.0198,0.9408 ± 0.0163,0.9308 ± 0.0241
test_precision,0.5579 ± 0.1062,0.5972 ± 0.1362,0.6236 ± 0.1298,0.6137 ± 0.1209,0.5551 ± 0.0898
train_precision,0.8769 ± 0.0468,0.8348 ± 0.0295,0.8567 ± 0.0486,0.8808 ± 0.0478,0.8766 ± 0.0409
test_recall,0.5644 ± 0.1377,0.5939 ± 0.1870,0.6428 ± 0.1690,0.6039 ± 0.1754,0.5256 ± 0.1195
train_recall,0.8851 ± 0.0511,0.8841 ± 0.0684,0.8695 ± 0.0622,0.8827 ± 0.0687,0.8513 ± 0.0623
test_f1,0.5522 ± 0.0938,0.5836 ± 0.1348,0.6201 ± 0.1236,0.5951 ± 0.1192,0.5321 ± 0.0858
train_f1,0.8788 ± 0.0245,0.8568 ± 0.0336,0.8602 ± 0.0254,0.8786 ± 0.0273,0.8615 ± 0.0281


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4809 ± 0.1634,0.5728 ± 0.1028,0.4850 ± 0.1449,0.5058 ± 0.1640,0.4972 ± 0.1613
train_roc_auc,0.9912 ± 0.0053,0.9546 ± 0.0143,0.9747 ± 0.0142,0.9823 ± 0.0087,0.9867 ± 0.0087
test_precision,0.5704 ± 0.0959,0.6130 ± 0.0840,0.5499 ± 0.0985,0.5528 ± 0.1092,0.5571 ± 0.0825
train_precision,0.9225 ± 0.0258,0.8304 ± 0.0347,0.8875 ± 0.0357,0.9073 ± 0.0301,0.9083 ± 0.0395
test_recall,0.7206 ± 0.1949,0.7467 ± 0.1251,0.6783 ± 0.1504,0.6728 ± 0.1897,0.7350 ± 0.1544
train_recall,0.9804 ± 0.0200,0.9479 ± 0.0347,0.9466 ± 0.0375,0.9701 ± 0.0265,0.9739 ± 0.0262
test_f1,0.6283 ± 0.1279,0.6675 ± 0.0805,0.6002 ± 0.1031,0.5995 ± 0.1299,0.6270 ± 0.0956
train_f1,0.9503 ± 0.0154,0.8847 ± 0.0270,0.9157 ± 0.0308,0.9373 ± 0.0210,0.9394 ± 0.0264


# ICANS >=2

In [108]:
y = targets["ae_summ_icans_highestgrade_v2"]
y = y.astype('float64')
y[y<2] = 0
y[y>=2] = 1

In [109]:
icans_2 = result_viewer(datasets, y, preprocessor,fs, models=models)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5651 ± 0.1420,0.5853 ± 0.1493,0.5953 ± 0.1264,0.5944 ± 0.1444,0.5456 ± 0.1744
train_roc_auc,0.8159 ± 0.0377,0.8121 ± 0.0340,0.8248 ± 0.0381,0.8391 ± 0.0418,0.8223 ± 0.0390
test_precision,0.4901 ± 0.2522,0.5073 ± 0.2283,0.4630 ± 0.1325,0.5171 ± 0.2297,0.4547 ± 0.2382
train_precision,0.7319 ± 0.0729,0.7412 ± 0.0661,0.7521 ± 0.0610,0.7796 ± 0.0669,0.7518 ± 0.0713
test_recall,0.3631 ± 0.1872,0.3500 ± 0.1766,0.3190 ± 0.1163,0.3595 ± 0.1754,0.3762 ± 0.1775
train_recall,0.5640 ± 0.0761,0.5397 ± 0.0849,0.5700 ± 0.0613,0.5890 ± 0.0801,0.5660 ± 0.0672
test_f1,0.3956 ± 0.1866,0.3987 ± 0.1736,0.3652 ± 0.1041,0.3968 ± 0.1621,0.3835 ± 0.1519
train_f1,0.6357 ± 0.0706,0.6217 ± 0.0737,0.6471 ± 0.0561,0.6682 ± 0.0670,0.6433 ± 0.0580


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4970 ± 0.1391,0.5876 ± 0.1619,0.5512 ± 0.1430,0.5465 ± 0.1185,0.4660 ± 0.1412
train_roc_auc,0.9385 ± 0.0188,0.9084 ± 0.0207,0.9200 ± 0.0216,0.9410 ± 0.0192,0.9440 ± 0.0129
test_precision,0.3849 ± 0.1709,0.4969 ± 0.2089,0.4152 ± 0.1632,0.4346 ± 0.1849,0.3650 ± 0.1858
train_precision,0.8484 ± 0.0467,0.8259 ± 0.0530,0.8430 ± 0.0612,0.8621 ± 0.0514,0.8588 ± 0.0534
test_recall,0.3702 ± 0.1674,0.4095 ± 0.1839,0.4226 ± 0.1819,0.3702 ± 0.1674,0.3369 ± 0.1676
train_recall,0.7845 ± 0.0862,0.7159 ± 0.0855,0.7328 ± 0.1110,0.7875 ± 0.0865,0.7969 ± 0.0740
test_f1,0.3641 ± 0.1448,0.4343 ± 0.1633,0.4091 ± 0.1529,0.3786 ± 0.1396,0.3442 ± 0.1629
train_f1,0.8108 ± 0.0396,0.7621 ± 0.0451,0.7754 ± 0.0543,0.8184 ± 0.0425,0.8225 ± 0.0309


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4635 ± 0.1683,0.5547 ± 0.1257,0.5091 ± 0.1238,0.5296 ± 0.1337,0.4847 ± 0.1514
train_roc_auc,0.9934 ± 0.0066,0.9732 ± 0.0111,0.9808 ± 0.0097,0.9873 ± 0.0070,0.9931 ± 0.0074
test_precision,0.2917 ± 0.3044,0.2983 ± 0.3258,0.3900 ± 0.3514,0.5292 ± 0.4191,0.3760 ± 0.3166
train_precision,0.9890 ± 0.0238,0.9634 ± 0.0365,0.9661 ± 0.0370,0.9791 ± 0.0232,0.9886 ± 0.0198
test_recall,0.1607 ± 0.1583,0.1179 ± 0.1229,0.1631 ± 0.1472,0.1524 ± 0.1177,0.1595 ± 0.1271
train_recall,0.8183 ± 0.0526,0.6128 ± 0.1008,0.7417 ± 0.0806,0.7570 ± 0.0736,0.8255 ± 0.0625
test_f1,0.1961 ± 0.1865,0.1594 ± 0.1585,0.2139 ± 0.1791,0.2189 ± 0.1575,0.2045 ± 0.1426
train_f1,0.8947 ± 0.0339,0.7426 ± 0.0754,0.8358 ± 0.0490,0.8515 ± 0.0470,0.8985 ± 0.0410


# CRS >=2

In [110]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('float64')
y[y<2] = 0
y[y>=2] = 1

In [111]:
crs_2 = result_viewer(datasets, y, preprocessor,fs, models=models)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6023 ± 0.1109,0.5114 ± 0.0978,0.6621 ± 0.1081,0.6318 ± 0.1040,0.5788 ± 0.1284
train_roc_auc,0.8714 ± 0.0375,0.8082 ± 0.0293,0.8596 ± 0.0272,0.8350 ± 0.0271,0.8482 ± 0.0351
test_precision,0.4804 ± 0.2010,0.3871 ± 0.2708,0.5282 ± 0.2695,0.4733 ± 0.1681,0.4292 ± 0.2825
train_precision,0.7571 ± 0.0620,0.7038 ± 0.0770,0.7600 ± 0.0554,0.7353 ± 0.0645,0.7472 ± 0.0853
test_recall,0.4333 ± 0.2000,0.2250 ± 0.1605,0.3833 ± 0.1979,0.3417 ± 0.1622,0.2833 ± 0.1756
train_recall,0.6146 ± 0.0683,0.4708 ± 0.0769,0.5833 ± 0.0632,0.5458 ± 0.0720,0.5375 ± 0.0902
test_f1,0.4500 ± 0.1941,0.2731 ± 0.1851,0.4188 ± 0.1987,0.3723 ± 0.1288,0.3293 ± 0.1926
train_f1,0.6772 ± 0.0599,0.5622 ± 0.0723,0.6576 ± 0.0475,0.6244 ± 0.0623,0.6221 ± 0.0819


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4996 ± 0.0964,0.4883 ± 0.1412,0.5682 ± 0.1166,0.5523 ± 0.1096,0.5515 ± 0.1266
train_roc_auc,0.9477 ± 0.0214,0.8915 ± 0.0170,0.9474 ± 0.0175,0.9359 ± 0.0204,0.9294 ± 0.0181
test_precision,0.3097 ± 0.1613,0.3079 ± 0.1982,0.4252 ± 0.1952,0.4354 ± 0.1625,0.3665 ± 0.1953
train_precision,0.8491 ± 0.0654,0.7616 ± 0.0432,0.8401 ± 0.0708,0.8627 ± 0.0796,0.8101 ± 0.0600
test_recall,0.3750 ± 0.2165,0.2917 ± 0.2033,0.4167 ± 0.1936,0.3750 ± 0.1656,0.3583 ± 0.2253
train_recall,0.8208 ± 0.1251,0.6937 ± 0.0737,0.8167 ± 0.1007,0.7312 ± 0.1025,0.7813 ± 0.0958
test_f1,0.3355 ± 0.1805,0.2940 ± 0.1956,0.3974 ± 0.1567,0.3874 ± 0.1468,0.3504 ± 0.1919
train_f1,0.8253 ± 0.0569,0.7224 ± 0.0356,0.8210 ± 0.0410,0.7835 ± 0.0563,0.7898 ± 0.0447


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6076 ± 0.1231,0.5242 ± 0.1060,0.6326 ± 0.1012,0.6121 ± 0.1019,0.5780 ± 0.1401
train_roc_auc,0.9839 ± 0.0102,0.9568 ± 0.0214,0.9654 ± 0.0115,0.9858 ± 0.0114,0.9787 ± 0.0093
test_precision,0.3203 ± 0.2656,0.1583 ± 0.3139,0.5308 ± 0.2780,0.4208 ± 0.2732,0.3450 ± 0.3305
train_precision,0.9569 ± 0.0454,0.9614 ± 0.0591,0.9400 ± 0.0451,0.9727 ± 0.0371,0.9585 ± 0.0442
test_recall,0.2083 ± 0.1891,0.0417 ± 0.0722,0.3000 ± 0.1354,0.1667 ± 0.1054,0.1750 ± 0.1441
train_recall,0.8083 ± 0.0624,0.5229 ± 0.0859,0.7125 ± 0.0732,0.7417 ± 0.0568,0.6833 ± 0.1057
test_f1,0.2383 ± 0.1892,0.0633 ± 0.1105,0.3624 ± 0.1378,0.2263 ± 0.1263,0.2147 ± 0.1684
train_f1,0.8740 ± 0.0338,0.6727 ± 0.0761,0.8076 ± 0.0493,0.8407 ± 0.0452,0.7914 ± 0.0712


# Treatment response
CMR vs others

In [112]:
y = targets["surv_bestresponse_car"]
y = y.astype('int8')
y[y != 1] = 0

In [113]:
outcome = result_viewer(datasets, y, preprocessor, fs, models=models)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5552 ± 0.1408,0.6354 ± 0.1306,0.5885 ± 0.1454,0.5965 ± 0.1569,0.6381 ± 0.1592
train_roc_auc,0.8640 ± 0.0468,0.8733 ± 0.0296,0.8691 ± 0.0351,0.8616 ± 0.0388,0.8969 ± 0.0244
test_precision,0.7650 ± 0.0653,0.7582 ± 0.0544,0.7641 ± 0.0578,0.7675 ± 0.0522,0.7768 ± 0.0563
train_precision,0.8399 ± 0.0253,0.8249 ± 0.0298,0.8408 ± 0.0248,0.8294 ± 0.0319,0.8483 ± 0.0281
test_recall,0.8205 ± 0.0881,0.8628 ± 0.0961,0.8702 ± 0.1112,0.8599 ± 0.1207,0.8522 ± 0.1034
train_recall,0.9522 ± 0.0156,0.9424 ± 0.0251,0.9600 ± 0.0235,0.9609 ± 0.0205,0.9541 ± 0.0199
test_f1,0.7896 ± 0.0648,0.8049 ± 0.0601,0.8109 ± 0.0696,0.8071 ± 0.0701,0.8099 ± 0.0656
train_f1,0.8922 ± 0.0150,0.8794 ± 0.0229,0.8962 ± 0.0184,0.8899 ± 0.0210,0.8977 ± 0.0175


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5760 ± 0.1362,0.5968 ± 0.1209,0.5680 ± 0.1263,0.6061 ± 0.1768,0.5737 ± 0.1510
train_roc_auc,0.9433 ± 0.0191,0.9246 ± 0.0221,0.9321 ± 0.0261,0.9360 ± 0.0215,0.9450 ± 0.0213
test_precision,0.7540 ± 0.0738,0.7712 ± 0.0663,0.7699 ± 0.0889,0.7908 ± 0.0674,0.7747 ± 0.0774
train_precision,0.9231 ± 0.0317,0.9106 ± 0.0403,0.9189 ± 0.0339,0.9223 ± 0.0350,0.9349 ± 0.0309
test_recall,0.7301 ± 0.1421,0.7494 ± 0.1352,0.6715 ± 0.1411,0.7455 ± 0.1312,0.7272 ± 0.1200
train_recall,0.9248 ± 0.0371,0.9081 ± 0.0425,0.9053 ± 0.0424,0.9093 ± 0.0405,0.9082 ± 0.0310
test_f1,0.7343 ± 0.0920,0.7547 ± 0.0906,0.7123 ± 0.1063,0.7621 ± 0.0916,0.7442 ± 0.0854
train_f1,0.9229 ± 0.0160,0.9078 ± 0.0160,0.9110 ± 0.0223,0.9145 ± 0.0183,0.9207 ± 0.0196


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5917 ± 0.1739,0.6622 ± 0.1450,0.6024 ± 0.1674,0.5938 ± 0.1604,0.5849 ± 0.1669
train_roc_auc,0.9881 ± 0.0092,0.9811 ± 0.0083,0.9890 ± 0.0080,0.9809 ± 0.0149,0.9910 ± 0.0053
test_precision,0.7455 ± 0.0253,0.7501 ± 0.0347,0.7465 ± 0.0273,0.7493 ± 0.0413,0.7510 ± 0.0467
train_precision,0.8504 ± 0.0283,0.8005 ± 0.0191,0.8189 ± 0.0303,0.8517 ± 0.0306,0.8426 ± 0.0259
test_recall,0.9606 ± 0.0533,0.9923 ± 0.0231,0.9763 ± 0.0437,0.9644 ± 0.0533,0.9724 ± 0.0376
train_recall,0.9980 ± 0.0059,1.0000 ± 0.0000,0.9990 ± 0.0043,1.0000 ± 0.0000,0.9981 ± 0.0058
test_f1,0.8388 ± 0.0292,0.8538 ± 0.0242,0.8455 ± 0.0279,0.8424 ± 0.0366,0.8469 ± 0.0387
train_f1,0.9181 ± 0.0175,0.8891 ± 0.0118,0.8997 ± 0.0183,0.9196 ± 0.0181,0.9136 ± 0.0159
